# UNSW-NB15 Intrusion Detection Project

This notebook runs the project's end-to-end smoke-test workflow: generate synthetic UNSW-NB15-shaped data, run EDA, train binary and multiclass models, and train a SMOTE multiclass run.

> **Important:** Synthetic data is only for checking that the pipeline works. Its metrics are meaningless and must not be presented as benchmark results. To use the real dataset, place `UNSW_NB15_training-set.csv` and `UNSW_NB15_testing-set.csv` in `data/` and set `TRAIN_CSV` and `TEST_CSV` below to those files.

Run cells from top to bottom with the project's Python environment selected as the notebook kernel.

In [1]:
from pathlib import Path
import os
import sys

# Find the project root even if Jupyter starts in a parent directory.
ROOT = Path.cwd().resolve()
while not (ROOT / "scripts" / "make_synthetic_data.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "scripts" / "make_synthetic_data.py").exists():
    raise FileNotFoundError("Could not find the project root containing scripts/make_synthetic_data.py")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data"
SYNTHETIC_DIR = DATA_DIR / "synthetic"
TRAIN_CSV = SYNTHETIC_DIR / "UNSW_NB15_training-set.csv"
TEST_CSV = SYNTHETIC_DIR / "UNSW_NB15_testing-set.csv"
print(f"Project root: {ROOT}")

Project root: C:\Users\adinath\Downloads\unsw_nb15_ids_project\unsw_nb15_ids_project


In [2]:
from importlib.metadata import PackageNotFoundError, version

required_packages = ["pandas", "numpy", "scikit-learn", "matplotlib", "joblib", "xgboost", "imbalanced-learn"]
for package in required_packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError as exc:
        raise RuntimeError(f"Missing dependency {package!r}; install the project dependencies with `pip install -r requirements.txt`.") from exc

pandas: 3.0.6
numpy: 2.5.3
scikit-learn: 1.9.1
matplotlib: 3.11.2
joblib: 1.6.0
xgboost: 3.4.1
imbalanced-learn: 0.14.2


In [3]:
import subprocess

# Create the local smoke-test dataset (synthetic scores are not meaningful).
subprocess.run(
    [sys.executable, "scripts/make_synthetic_data.py", "--out", str(SYNTHETIC_DIR), "--n-train", "6000", "--n-test", "3000"],
    cwd=ROOT,
    check=True,
)
print(f"Training data: {TRAIN_CSV}")
print(f"Testing data:  {TEST_CSV}")

Training data: C:\Users\adinath\Downloads\unsw_nb15_ids_project\unsw_nb15_ids_project\data\synthetic\UNSW_NB15_training-set.csv
Testing data:  C:\Users\adinath\Downloads\unsw_nb15_ids_project\unsw_nb15_ids_project\data\synthetic\UNSW_NB15_testing-set.csv


In [4]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
print(f"Training shape: {train_df.shape}; testing shape: {test_df.shape}")
display(train_df.head())
display(train_df["attack_cat"].value_counts().rename("training rows").to_frame())

Training shape: (6000, 45); testing shape: (3000, 45)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,2.727122,icmp,ssh,REQ,21.982817,33.390719,2.284379,18.784813,16.410292,...,3.891989,3.561478,0,1.102984,7.109568,9.345816,6.685964,0,Normal,0
1,2,4.624404,tcp,http,CON,9.890114,26.468148,1.512112,5.509259,11.082536,...,56.539803,0.945802,0,19.440749,12.850123,5.757645,2.759837,0,Generic,1
2,3,4.775505,udp,http,CON,17.561915,39.784084,1.086224,2.048547,12.943052,...,4.700376,1.390735,0,7.726545,32.807307,7.731970,1.880632,0,Generic,1
3,4,10.602175,arp,smtp,CON,31.033584,40.583935,1.778899,0.907915,11.335559,...,1.958331,7.943718,0,1.477893,0.807763,4.298819,1.519775,1,Exploits,1
4,5,18.976267,sctp,dns,CON,53.068213,71.704279,2.338438,10.854856,6.837440,...,30.163373,6.682207,0,14.651804,5.002381,0.275552,11.990656,0,Normal,0


,training rows
attack_cat,
Normal,1979
Generic,1345
Exploits,1129
Fuzzers,571
DoS,435
Reconnaissance,371
Backdoor,65
Analysis,64
Shellcode,33


## Exploratory data analysis

The project writes its EDA tables and figures under `reports/results/eda/` and `reports/figures/eda/`.

In [5]:
from src.eda import main as run_eda

run_eda(["--train", str(TRAIN_CSV)])

 rows  columns  missing_cells  infinite_cells  duplicate_rows_ignoring_id
 6000       45             24              12                           0
EDA figures -> C:\Users\adinath\Downloads\unsw_nb15_ids_project\unsw_nb15_ids_project\reports\figures\eda
EDA tables  -> C:\Users\adinath\Downloads\unsw_nb15_ids_project\unsw_nb15_ids_project\reports\results\eda


## Binary detection: Normal vs Attack

Trains the five configured models and saves metrics, plots, and model artifacts under the `binary_notebook` output directories.

In [6]:
from src.train import main as run_training

common_paths = ["--train", str(TRAIN_CSV), "--test", str(TEST_CSV)]
run_training(common_paths + ["--task", "binary", "--tag", "notebook"])

Train (6000, 42) | Test (3000, 42) | task=binary
Train class counts: {'Normal': np.int64(1979), 'Attack': np.int64(4021)}

=== LogisticRegression ===
  accuracy=0.7497  macro_f1=0.7473  attack_detection_rate=0.6361  false_alarm_rate=0.0249  roc_auc=0.9195

=== DecisionTree ===
  accuracy=0.8067  macro_f1=0.7829  attack_detection_rate=0.8551  false_alarm_rate=0.2896  roc_auc=0.7828

=== RandomForest ===
  accuracy=0.9187  macro_f1=0.9093  attack_detection_rate=0.9328  false_alarm_rate=0.1095  roc_auc=0.9745

=== XGBoost ===
  accuracy=0.9223  macro_f1=0.9153  attack_detection_rate=0.9103  false_alarm_rate=0.0537  roc_auc=0.9789

=== MLP ===
  accuracy=0.9163  macro_f1=0.9092  attack_detection_rate=0.9003  false_alarm_rate=0.0517  roc_auc=0.9766

             model  accuracy  macro_precision  macro_recall  macro_f1  weighted_f1  attack_detection_rate  false_alarm_rate  roc_auc  train_seconds  inference_ms_per_1000_flows
           XGBoost    0.9223           0.9064        0.9283    0.915

,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,attack_detection_rate,false_alarm_rate,roc_auc,train_seconds,inference_ms_per_1000_flows
3,XGBoost,0.922333,0.906358,0.928272,0.915291,0.923351,0.910276,0.053731,0.978899,3.607823,12.956533
2,RandomForest,0.918667,0.906984,0.911690,0.909258,0.918900,0.932832,0.109453,0.974516,3.331453,79.649967
4,MLP,0.916333,0.899559,0.924255,0.909161,0.917584,0.900251,0.051741,0.976587,2.575384,10.049333
1,DecisionTree,0.806667,0.783073,0.782793,0.782932,0.806619,0.855138,0.289552,0.782793,0.450408,10.230733
0,LogisticRegression,0.749667,0.777562,0.805607,0.747322,0.755354,0.636090,0.024876,0.919461,0.097882,4.707467


## Multiclass attack-family classification

Trains the configured classifiers to distinguish the attack categories.

In [7]:
run_training(common_paths + ["--task", "multiclass", "--tag", "notebook"])

Train (6000, 42) | Test (3000, 42) | task=multiclass
Train class counts: {'Analysis': np.int64(64), 'Backdoor': np.int64(65), 'DoS': np.int64(435), 'Exploits': np.int64(1129), 'Fuzzers': np.int64(571), 'Generic': np.int64(1345), 'Normal': np.int64(1979), 'Reconnaissance': np.int64(371), 'Shellcode': np.int64(33), 'Worms': np.int64(8)}

=== LogisticRegression ===
  accuracy=0.9633  macro_f1=0.9354  attack_detection_rate=0.9644  false_alarm_rate=0.0338  roc_auc=0.9994

=== DecisionTree ===
  accuracy=0.6850  macro_f1=0.5802  attack_detection_rate=0.7794  false_alarm_rate=0.2886  roc_auc=0.7505

=== RandomForest ===
  accuracy=0.9340  macro_f1=0.7794  attack_detection_rate=0.9273  false_alarm_rate=0.0408  roc_auc=0.9977

=== XGBoost ===
  accuracy=0.9330  macro_f1=0.7875  attack_detection_rate=0.9258  false_alarm_rate=0.0348  roc_auc=0.9944

=== MLP ===
  accuracy=0.9280  macro_f1=0.7751  attack_detection_rate=0.9078  false_alarm_rate=0.0269  roc_auc=0.9982

             model  accuracy  

,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,attack_detection_rate,false_alarm_rate,roc_auc,train_seconds,inference_ms_per_1000_flows
0,LogisticRegression,0.963333,0.950227,0.926441,0.935449,0.962712,0.964411,0.033831,0.999375,0.356354,9.302200
3,XGBoost,0.933000,0.878974,0.743016,0.787463,0.930014,0.925815,0.034826,0.994415,17.476265,34.834067
2,RandomForest,0.934000,0.970663,0.719651,0.779414,0.930432,0.927318,0.040796,0.997706,2.934121,97.459467
4,MLP,0.928000,0.875718,0.721904,0.775092,0.923643,0.907769,0.026866,0.998217,3.620215,10.814333
1,DecisionTree,0.685000,0.654883,0.542770,0.580207,0.680528,0.779449,0.288557,0.750474,0.929205,7.476500


## SMOTE imbalance-handling run

SMOTE is applied to training data only. This run may take longer and produces a separate set of model artifacts and reports.

In [8]:
run_training(common_paths + ["--task", "multiclass", "--smote", "--tag", "notebook_smote"])

Train (6000, 42) | Test (3000, 42) | task=multiclass
Train class counts: {'Analysis': np.int64(64), 'Backdoor': np.int64(65), 'DoS': np.int64(435), 'Exploits': np.int64(1129), 'Fuzzers': np.int64(571), 'Generic': np.int64(1345), 'Normal': np.int64(1979), 'Reconnaissance': np.int64(371), 'Shellcode': np.int64(33), 'Worms': np.int64(8)}

=== LogisticRegression ===
  accuracy=0.9530  macro_f1=0.9145  attack_detection_rate=0.9564  false_alarm_rate=0.0468  roc_auc=0.9990

=== DecisionTree ===
  accuracy=0.6873  macro_f1=0.5049  attack_detection_rate=0.7749  false_alarm_rate=0.2607  roc_auc=0.7237

=== RandomForest ===
  accuracy=0.9153  macro_f1=0.7043  attack_detection_rate=0.8812  false_alarm_rate=0.0139  roc_auc=0.9980

=== XGBoost ===
  accuracy=0.9317  macro_f1=0.8402  attack_detection_rate=0.9293  false_alarm_rate=0.0408  roc_auc=0.9932

=== MLP ===
  accuracy=0.9380  macro_f1=0.8694  attack_detection_rate=0.9258  false_alarm_rate=0.0318  roc_auc=0.9985

             model  accuracy  

,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,attack_detection_rate,false_alarm_rate,roc_auc,train_seconds,inference_ms_per_1000_flows
0,LogisticRegression,0.953000,0.939171,0.899579,0.914458,0.952183,0.956391,0.046766,0.998970,2.594359,5.955533
4,MLP,0.938000,0.924386,0.840974,0.869361,0.935663,0.925815,0.031841,0.998475,37.664058,10.497267
3,XGBoost,0.931667,0.891450,0.821400,0.840204,0.929835,0.929323,0.040796,0.993229,72.839578,43.562333
2,RandomForest,0.915333,0.875990,0.649690,0.704265,0.909315,0.881203,0.013930,0.998009,74.463332,81.442867
1,DecisionTree,0.687333,0.535025,0.488539,0.504871,0.687135,0.774937,0.260697,0.723710,24.296396,10.563667


## Results and generated artifacts

The following tables contain synthetic-data smoke-test metrics only. For real benchmark results, use the official UNSW-NB15 partitioned CSV files and rerun the notebook.

In [9]:
result_files = {
    "Binary": ROOT / "reports" / "results" / "binary_notebook" / "metrics_summary.csv",
    "Multiclass": ROOT / "reports" / "results" / "multiclass_notebook" / "metrics_summary.csv",
    "Multiclass with SMOTE": ROOT / "reports" / "results" / "multiclass_notebook_smote" / "metrics_summary.csv",
}
for run_name, result_path in result_files.items():
    print(run_name)
    if not result_path.exists():
        raise FileNotFoundError(f"Expected results were not created: {result_path}")
    display(pd.read_csv(result_path))

print("\nArtifacts:")
print(f"Models:       {ROOT / 'models'}")
print(f"Metrics:      {ROOT / 'reports' / 'results'}")
print(f"Figures:      {ROOT / 'reports' / 'figures'}")
print("Launch the interactive demo from the project root with: streamlit run app/streamlit_app.py")

Binary


,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,attack_detection_rate,false_alarm_rate,roc_auc,train_seconds,inference_ms_per_1000_flows
0,XGBoost,0.922333,0.906358,0.928272,0.915291,0.923351,0.910276,0.053731,0.978899,3.607823,12.956533
1,RandomForest,0.918667,0.906984,0.911690,0.909258,0.918900,0.932832,0.109453,0.974516,3.331453,79.649967
2,MLP,0.916333,0.899559,0.924255,0.909161,0.917584,0.900251,0.051741,0.976587,2.575384,10.049333
3,DecisionTree,0.806667,0.783073,0.782793,0.782932,0.806619,0.855138,0.289552,0.782793,0.450408,10.230733
4,LogisticRegression,0.749667,0.777562,0.805607,0.747322,0.755354,0.636090,0.024876,0.919461,0.097882,4.707467


Multiclass


,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,attack_detection_rate,false_alarm_rate,roc_auc,train_seconds,inference_ms_per_1000_flows
0,LogisticRegression,0.963333,0.950227,0.926441,0.935449,0.962712,0.964411,0.033831,0.999375,0.356354,9.302200
1,XGBoost,0.933000,0.878974,0.743016,0.787463,0.930014,0.925815,0.034826,0.994415,17.476265,34.834067
2,RandomForest,0.934000,0.970663,0.719651,0.779414,0.930432,0.927318,0.040796,0.997706,2.934121,97.459467
3,MLP,0.928000,0.875718,0.721904,0.775092,0.923643,0.907769,0.026866,0.998217,3.620215,10.814333
4,DecisionTree,0.685000,0.654883,0.542770,0.580207,0.680528,0.779449,0.288557,0.750474,0.929205,7.476500


Multiclass with SMOTE


,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,attack_detection_rate,false_alarm_rate,roc_auc,train_seconds,inference_ms_per_1000_flows
0,LogisticRegression,0.953000,0.939171,0.899579,0.914458,0.952183,0.956391,0.046766,0.998970,2.594359,5.955533
1,MLP,0.938000,0.924386,0.840974,0.869361,0.935663,0.925815,0.031841,0.998475,37.664058,10.497267
2,XGBoost,0.931667,0.891450,0.821400,0.840204,0.929835,0.929323,0.040796,0.993229,72.839578,43.562333
3,RandomForest,0.915333,0.875990,0.649690,0.704265,0.909315,0.881203,0.013930,0.998009,74.463332,81.442867
4,DecisionTree,0.687333,0.535025,0.488539,0.504871,0.687135,0.774937,0.260697,0.723710,24.296396,10.563667



Artifacts:
Models:       C:\Users\adinath\Downloads\unsw_nb15_ids_project\unsw_nb15_ids_project\models
Metrics:      C:\Users\adinath\Downloads\unsw_nb15_ids_project\unsw_nb15_ids_project\reports\results
Figures:      C:\Users\adinath\Downloads\unsw_nb15_ids_project\unsw_nb15_ids_project\reports\figures
Launch the interactive demo from the project root with: streamlit run app/streamlit_app.py
